In [12]:
import pandas as pd
import geopandas as gpd



In [13]:
stations_gdf = gpd.read_file("../raw/velib-emplacement-des-stations.geojson")
iris_gdf = gpd.read_file("../raw/iris_raw_20260427.geojson")

In [14]:
stations_gdf.head()

,stationcode,name,capacity,station_opening_hours,geometry
0,11104,Charonne - Robert et Sonia Delaunay,20,None,POINT (2.39257 48.85591)
1,22207,Rond-point du Dr Albert Schweitzer,22,None,POINT (2.31439 48.78981)
2,44010,8 Mai 1945 - 10 Juillet 1940,30,None,POINT (2.3979 48.78457)
3,31009,Place Du Général De Gaulle,26,None,POINT (2.43299 48.86876)
4,21312,Lucie Aubrac - Fort,30,None,POINT (2.27118 48.81814)


In [16]:
iris_paris = iris_gdf[iris_gdf["dep"] == "75"].copy()

In [8]:
iris_paris = iris_paris.to_crs(stations_gdf.crs)

In [18]:
stations_iris = gpd.sjoin(
    stations_gdf,
    iris_paris,
    how="inner",
    predicate="within"
)

In [19]:
stations_iris.head()

,stationcode,name,capacity,station_opening_hours,geometry,index_right,dep,insee_com,nom_com,iris,code_iris,nom_iris,typ_iris,geo_point_2d,id
0,11104,Charonne - Robert et Sonia Delaunay,20,None,POINT (2.39257 48.85591),4798,75,75111,Paris 11e Arrondissement,4411,751114411,Sainte-Marguerite 11,H,"{'lon': 2.3922563562623482, 'lat': 48.85457318...",IRIS____0000000751114411
5,15133,Saint Lambert - Blomet,27,None,POINT (2.29306 48.83659),1675,75,75115,Paris 15e Arrondissement,5713,751155713,Saint-Lambert 13,H,"{'lon': 2.294073338823648, 'lat': 48.837188543...",IRIS____0000000751155713
7,8050,Boétie - Ponthieu,45,None,POINT (2.30768 48.87142),2151,75,75108,Paris 8e Arrondissement,3004,751083004,Faubourg du Roule 4,A,"{'lon': 2.308904326468502, 'lat': 48.872113745...",IRIS____0000000751083004
8,15010,Square Cambronne,59,None,POINT (2.30257 48.84757),3825,75,75115,Paris 15e Arrondissement,5811,751155811,Necker 11,H,"{'lon': 2.3033426848325282, 'lat': 48.84660474...",IRIS____0000000751155811
11,17122,Bridaine - Lamandé,29,None,POINT (2.32062 48.88621),1925,75,75117,Paris 17e Arrondissement,6705,751176705,Batignolles 5,H,"{'lon': 2.3208513967712916, 'lat': 48.88557700...",IRIS____0000000751176705


In [21]:
stations_iris = stations_iris[[
    "stationcode",
    "name",
    "capacity",
    "geometry",
    "iris",
    "geo_point_2d",
    "code_iris",
    "nom_iris"
]]

In [22]:
stations_iris.to_parquet("../silver/silver_velib_iris.parquet")

In [24]:
df_aggregated = (stations_iris.groupby("iris").agg(velib_station=("stationcode","count"),capacity=("capacity","sum")).reset_index())

In [25]:
df_silver = pd.read_parquet("../silver/silver_velib_iris.parquet")

In [26]:
df_aggregated = (df_silver.groupby(["code_iris","nom_iris"]).agg(velib_station=("stationcode","count"),capacity=("capacity","sum")).reset_index())

In [27]:
df_aggregated

,code_iris,nom_iris,velib_station,capacity
0,751010101,Saint-Germain l'Auxerrois 1,1,22
1,751010102,Saint-Germain l'Auxerrois 2,1,17
2,751010103,Saint-Germain l'Auxerrois 3,1,28
3,751010201,Les Halles 1,3,100
4,751010202,Les Halles 2,4,92
...,...,...,...,...
628,751208018,Charonne 18,1,33
629,751208019,Charonne 19,1,23
630,751208021,Charonne 21,3,72
631,751208023,Charonne 23,2,51


In [28]:
df_aggregated.to_parquet("../gold/capacity_velib_iris.parquet")